# Reviewer reproducibility check

Public-safe smoke checks using only synthetic records. The same package functions are used by the full pipeline.

In [ ]:
from pathlib import Path
import sys, platform, tempfile
import pandas as pd, networkx as nx
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import fusion_pipeline
print({'python': platform.python_version(), 'pandas': pd.__version__, 'networkx': nx.__version__})

In [ ]:
from fusion_pipeline.config import validate_method_presets
assert validate_method_presets(ROOT / 'config')['status'] == 'pass'

In [ ]:
from fusion_pipeline.wos_parser import parse_wos_file
text = 'PT J\nAU Smith, AB\n   Jones, CD\nTI Synthetic fusion title\nDT Article\nPY 1986\nUT SYNTHETIC\nER\n'
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / 'synthetic.txt'; path.write_text(text)
    parsed = parse_wos_file(path)
assert len(parsed) == 1 and parsed[0]['authors_wos'] == ['Smith, AB', 'Jones, CD']

In [ ]:
from fusion_pipeline.corpus import derive_analytical_corpus, validate_identity_subset
records = pd.DataFrame([
 {'record_id':'a','document_types':['Article'],'authors_normalized':['SMITH, AB'],'authors_disambiguated':['SMITH, AB'],'authors_network':['SMITH, AB']},
 {'record_id':'e','document_types':['Editorial Material'],'authors_normalized':['JONES, CD'],'authors_disambiguated':['JONES, CD'],'authors_network':['JONES, CD']},
 {'record_id':'x','document_types':['Meeting Abstract'],'authors_normalized':['LEE, EF'],'authors_disambiguated':['LEE, EF'],'authors_network':['LEE, EF']},])
analytical, audit, private = derive_analytical_corpus(records, {'x'})
assert analytical.record_id.tolist() == ['a']
assert validate_identity_subset(records, analytical)['changed_mappings'] == 0

In [ ]:
from fusion_pipeline.networks import prepare_network_dataframe, build_coauthorship_graph, largest_component_graph, louvain_partition
papers = prepare_network_dataframe(pd.DataFrame([
 {'record_id':'p1','year':1986,'authors_network':['A','B','C'],'document_types':['Article']},
 {'record_id':'p2','year':1986,'authors_network':['C','D'],'document_types':['Article']}]))
graph = build_coauthorship_graph(papers); lcc = largest_component_graph(graph)
assert graph['A']['B']['fractional_weight'] == 0.5
assert louvain_partition(lcc, seed=42, weight='paper_count') == louvain_partition(lcc, seed=42, weight='paper_count')

In [ ]:
from fusion_pipeline.socio_epistemic import tokenize_title, community_titleword_profiles
assert tokenize_title('Electron-cyclotron heating', set(), {}) == ['electron-cyclotron','heating']
social = (papers, graph, lcc, {'A':0,'B':0,'C':1,'D':1})
profiles = community_titleword_profiles(papers.assign(title=['alpha','beta']), 1986, set(), {}, social=social)
assignment = profiles['paper_assignments'].iloc[0]
assert assignment.n_communities == 2 and '0:0.5' in assignment.community_weights and '1:0.5' in assignment.community_weights

In [ ]:
from fusion_pipeline.normalization import build_affiliation_table
jet = build_affiliation_table([{'record_id':'r','year':1986,'affiliations':[], 'reprint_address_one_line':'Smith, AB (reprint author), JET Joint Undertaking, Abingdon, England'}])
assert jet.iloc[0].country_normalized == 'UK' and jet.iloc[0].network_country == 'JET'

In [ ]:
from fusion_pipeline.corpus import require_licensed_input, LicensedInputMissingError
try:
    require_licensed_input(None, 'synthetic licensed input')
except LicensedInputMissingError as exc:
    assert 'No Web of Science record-level data are bundled' in str(exc)
else:
    raise AssertionError('missing licensed input must fail clearly')
print('All reviewer smoke checks passed.')